In [1]:
import numpy as np

In [13]:
#Generate input
np.random.seed(42)

conv_weights = np.random.randn(8,3,3,3).astype(np.float32)
conv_weights.shape

(8, 3, 3, 3)

In [14]:
#Creating range imbalance
conv_weights[0] *= 0.1
conv_weights[1] *= 0.5
conv_weights[6] *= 5.0
conv_weights[7] *= 10.0

In [47]:
def per_tensor_quantize(tensor):
    scale = (np.max(np.abs(tensor)))/127
    #Since we are using symmetric quantization zp = 0
    quant_tensor_rounded = np.round(tensor/scale)
    quant_tensor_clip = np.clip(quant_tensor_rounded,-127,127)
    quant_tensor = quant_tensor_clip.astype(np.int8)
    dequant_tensor = quant_tensor_clip * scale
    return (quant_tensor, dequant_tensor, scale)

In [48]:
q_tensor,dq_tensor,scale = per_tensor_quantize(conv_weights)
mae = np.mean(np.abs(conv_weights - dq_tensor))
print("Scale: ",scale)
print("MAE: ",mae)

Scale:  0.30336466
MAE:  0.07114315


In [65]:
for i in range(conv_weights.shape[0]):
    print(f"Per tensor quantization on channel {i}")
    x_min = np.min(conv_weights[i])
    x_max = np.max(conv_weights[i])
    mae = np.mean(np.abs(conv_weights[i] - dq_tensor[i]))
    print("MAE: ",mae)
    print(f"Range: {x_min},{x_max}")

Per tensor quantization on channel 0
MAE:  0.07146436
Range: -0.1913280338048935,0.15792128443717957
Per tensor quantization on channel 1
MAE:  0.07256411
Range: -0.979835033416748,0.9261391162872314
Per tensor quantization on channel 2
MAE:  0.072174944
Range: -2.6197450160980225,1.5646436214447021
Per tensor quantization on channel 3
MAE:  0.071593724
Range: -1.4635149240493774,1.886185884475708
Per tensor quantization on channel 4
MAE:  0.07057091
Range: -1.9187712669372559,2.4632420539855957
Per tensor quantization on channel 5
MAE:  0.06893915
Range: -1.6074832677841187,1.8657745122909546
Per tensor quantization on channel 6
MAE:  0.07780033
Range: -5.354462146759033,13.600845336914062
Per tensor quantization on channel 7
MAE:  0.06403764
Range: -15.148472785949707,38.527313232421875


In [73]:
def per_channel_quantize(tensor):
    scales = []
    quantized_tensor = []
    dequantized_tensor = []
    for i in range(tensor.shape[0]):
        channel = tensor[i] # i represents the channel index
        scale = (np.max(np.abs(channel))) / 127
        quant_channel_rounded = np.round(channel/scale)
        quant_channel_clip = np.clip(quant_channel_rounded,-127,127)
        quant_channel = quant_channel_clip.astype(np.int8)
        dequant_channel = quant_channel_clip * scale
        quantized_tensor.append(quant_channel)
        dequantized_tensor.append(dequant_channel)
        scales.append(scale)
    quantized_tensor = np.array(quantized_tensor)
    dequantized_tensor = np.array(dequantized_tensor)
    scales = np.array(scales)
    return (quantized_tensor, dequantized_tensor, scales)

In [76]:
for i in range(conv_weights.shape[0]):
    print(f"Per-channel quantization on channel {i}")
    q_tensor,dq_tensor,scale = per_channel_quantize(conv_weights)
    x_min = np.min(conv_weights[i])
    x_max = np.max(conv_weights[i])
    channel_scale = scale[i]
    mae = np.mean(np.abs(conv_weights[i] - dq_tensor[i]))
    print("Scale: ",channel_scale)
    print("MAE: ",mae)
    print(f"Range: {x_min},{x_max}")

Per-channel quantization on channel 0
Scale:  0.00150652
MAE:  0.00032642912
Range: -0.1913280338048935,0.15792128443717957
Per-channel quantization on channel 1
Scale:  0.0077152364
MAE:  0.0016606675
Range: -0.979835033416748,0.9261391162872314
Per-channel quantization on channel 2
Scale:  0.020627914
MAE:  0.005372616
Range: -2.6197450160980225,1.5646436214447021
Per-channel quantization on channel 3
Scale:  0.014851857
MAE:  0.0037305246
Range: -1.4635149240493774,1.886185884475708
Per-channel quantization on channel 4
Scale:  0.019395607
MAE:  0.003995452
Range: -1.9187712669372559,2.4632420539855957
Per-channel quantization on channel 5
Scale:  0.014691138
MAE:  0.0039005855
Range: -1.6074832677841187,1.8657745122909546
Per-channel quantization on channel 6
Scale:  0.10709327
MAE:  0.027713789
Range: -5.354462146759033,13.600845336914062
Per-channel quantization on channel 7
Scale:  0.30336466
MAE:  0.06403764
Range: -15.148472785949707,38.527313232421875
